# BSM L07G — App provenance, runtime trust, backup/migracja (Android)

## Tryb pracy
Pracujesz głównie w **Android Studio** na projekcie `student/apps/lesson_g_app`.
Ten notebook to:
- opis teoretyczny (dlaczego to robimy),
- instrukcja krok po kroku (co otworzyć, gdzie kliknąć, jak uruchomić testy),
- formularz i wysyłka odpowiedzi.

## Zasady oddawania
- **G01**: odpowiedź wysyła aplikacja automatycznie (w notebooku nie ma komórki wysyłki).
- **G02-G04**: wypełniasz pola w formularzu i uruchamiasz komórkę „Wyślij”.

## Starter projektu
Ścieżka w repo (relatywnie): `student/apps/lesson_g_app`


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


# G01 — Manifest i audyt prywatności (auto-submit z aplikacji)

## Cel
Nauczyć się odróżniać:
- deklaracje w `AndroidManifest.xml` (co aplikacja *może* robić),
- realne użycie w kodzie (co aplikacja *robi*),
- prośby runtime (kiedy użytkownik widzi prompt),
- minimalizację (czy da się z mniejszym zakresem uprawnień).

## Część teoretyczna
- Uprawnienie w manifeście nie oznacza automatycznie, że aplikacja dostanie dostęp.
- Dla wielu uprawnień Android wymaga osobnej zgody użytkownika w runtime.
- Dobre praktyki:
  1. prosić o uprawnienie dopiero gdy funkcja jest potrzebna,
  2. uzasadniać w UI, po co to jest,
  3. mieć fallback (aplikacja nie powinna się „wywracać” po odmowie).

## Co masz zrobić (krok po kroku)
1. Otwórz `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik: `app/src/main/AndroidManifest.xml`.
1. Przejrzyj wszystkie wpisy `<uses-permission ...>`.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Znajdź w UI sekcję „Student / Task 1” i zobacz, jakie uprawnienia aplikacja żąda po kliknięciu „Request permissions”.
1. Zrób mapę: każde uprawnienie -> jaka funkcja je wykorzystuje (mapa, zdjęcia, kamera, internet).
1. Sprawdź zachowanie fallback:
- co aplikacja pokazuje, gdy nie ma lokalizacji,
- co pokazuje, gdy brak dostępu do zdjęć,
- co pokazuje, gdy nie ma kamery.

## Jak zaliczasz (auto-submit)
1. Uruchom aplikację.
1. Wpisz swoje `Student ID`.
1. Kliknij „Request permissions” i przejdź cały przepływ.
1. Gdy ID + wymagane uprawnienia są OK, aplikacja sama wyśle odpowiedź dla `G01`.


# G02 — APK / bundle provenance check (tożsamość builda)

## Część teoretyczna
W tym zadaniu chodzi o zrobienie „sprawdzenia pochodzenia” z perspektywy aplikacji:
- czy ten build jest podpisany oczekiwanym kluczem (tożsamość wydawcy),
- czy wygląda na przepakowany / zmodyfikowany (tampering/repackage),
- i czy rozumiesz różnicę między zaufaniem instalacyjnym (system) a decyzją w runtime (aplikacja/backend).

Kluczowa intuicja:
- „To samo `packageName`” nie gwarantuje „ta sama aplikacja”.
- Podpis (certyfikat) jest częścią tożsamości.

## Co masz zrobić (krok po kroku)
1. Otwórz projekt `student/apps/lesson_g_app`.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
   Zobacz strukturę `ProvenanceState` i warunki na kod `G02`.
1. Zaimplementuj w aplikacji przepływ, który potrafi wyznaczyć `ProvenanceState` na podstawie realnych danych z systemu.
   W praktyce to oznacza:
- pobranie informacji o podpisie z `PackageManager`,
- wyznaczenie identyfikatora podpisu (np. SHA-256 certyfikatu),
- porównanie z wartością oczekiwaną (podaną w labie / teście),
- ustawienie `buildLooksTampered` w oparciu o warunki, które lab definiuje (np. brak zgodności podpisu, nielogiczny zestaw informacji, itp.).

## Co konkretnie otworzyć / gdzie kliknąć
1. Android Studio: otwórz okno Gradle (`View` -> `Tool Windows` -> `Gradle`).
1. Uruchom testy jednostkowe:
- `lesson_g_app` -> `app` -> `Tasks` -> `verification` -> `testDebugUnitTest`
1. Alternatywnie uruchom evidence w terminalu w katalogu `student/apps/lesson_g_app`:
- `./gradlew :app:bsmEvidence`

## Dokumentacja (konkretne API)
- `PackageManager`: https://developer.android.com/reference/android/content/pm/PackageManager
- `PackageManager.getPackageInfo(...)`: https://developer.android.com/reference/android/content/pm/PackageManager#getPackageInfo(java.lang.String,int)
- Flaga: `PackageManager.GET_SIGNING_CERTIFICATES`
- `PackageInfo.signingInfo`: https://developer.android.com/reference/android/content/pm/PackageInfo#signingInfo
- `SigningInfo.getApkContentsSigners()`: https://developer.android.com/reference/android/content/pm/SigningInfo#getApkContentsSigners()
- `Signature.toByteArray()`: https://developer.android.com/reference/android/content/pm/Signature#toByteArray()
- `MessageDigest` (SHA-256): https://developer.android.com/reference/java/security/MessageDigest

## Co wpisać do odpowiedzi
W odpowiedzi wpisujesz **5-znakowy kod** z `:app:bsmEvidence`.


In [ ]:
#@title G02 — Formularz odpowiedzi
code_g02 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g02.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G02", final_answer)


# G03 — Integrity-gated backend request (zaufanie w runtime + binding)

## Część teoretyczna
Provenance check (G02) odpowiada na pytanie: „czy to jest oczekiwany build?”.
W G03 idziesz krok dalej: nawet jeśli build wygląda OK, backend powinien wymagać sygnału zaufania i powiązania żądania z tożsamością aplikacji.

W tym labie (zgodnie ze starterem/testami) modelujesz to przez `IntegrityState`:
- `verdict` (np. `ALLOW`),
- `appPackageNameMatches` (kontrola tożsamości),
- `requestIsBoundToAppIdentity` (binding żądania do tożsamości builda).

## Co masz zrobić (krok po kroku)
1. Otwórz: `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
1. Otwórz: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
   Znajdź funkcję `submitAnswer(...)`.
1. Zaimplementuj mechanizm, w którym:
- najpierw wyznaczasz `IntegrityState`,
- potem decydujesz, czy wolno wykonać request,
- jeśli nie wolno: robisz bezpieczny fallback (nie wysyłasz nic) i pokazujesz komunikat.

## Co uruchomić
- `./gradlew :app:bsmEvidence` (w `student/apps/lesson_g_app`) i wklej kod.

## Dokumentacja (konkretne elementy używane w starterze)
- `HttpURLConnection`: https://developer.android.com/reference/java/net/HttpURLConnection
- `URL`: https://developer.android.com/reference/java/net/URL

## Co wpisać do odpowiedzi
Wklej **5-znakowy kod** z `:app:bsmEvidence`.


In [ ]:
#@title G03 — Formularz odpowiedzi
code_g03 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g03.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G03", final_answer)


# G04 — Backup/migracja: higiena sekretów

## Część teoretyczna
Backup i migracja mogą przenosić stan aplikacji między urządzeniami. To jest ryzykowne dla sekretów.
Twoim zadaniem jest ustawić politykę tak, aby dane wrażliwe nie migrowały w sposób niezamierzony.

## Co masz zrobić (krok po kroku)
1. Otwórz: `app/src/main/AndroidManifest.xml`.
1. Sprawdź `android:allowBackup`.
1. Zdecyduj:
- albo wyłączasz backup,
- albo zostawiasz backup i dodajesz reguły wykluczające wrażliwe dane.
1. Jeśli robisz reguły:
- dodaj plik XML w `app/src/main/res/xml/`,
- podepnij go w `<application ...>`.

## Dokumentacja (konkretne atrybuty)
- `android:allowBackup`: https://developer.android.com/guide/topics/manifest/application-element#allowbackup
- `android:fullBackupContent`: https://developer.android.com/guide/topics/manifest/application-element#fullBackupContent
- `android:dataExtractionRules`: https://developer.android.com/guide/topics/manifest/application-element#dataExtractionRules

## Co wpisać do odpowiedzi
Wklej 5-znakową wartość sekretu dla G04 (zgodnie ze starterem). Podpowiedź: szukaj `TASK_4_SECRET_...` w `MainActivity.kt`.


In [ ]:
#@title G04 — Formularz odpowiedzi
secret_g04 = ""  #@param {type:"string"}

final_answer = prepare_answer(secret_g04.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G04", final_answer)
